## 1. 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('ggplot')
plt.rc('font', family='Malgun Gothic') # Windows 한글 폰트 설정

In [ ]:
try:
    # train = pd.read_csv('data/train.csv')
    # test = pd.read_csv('data/test.csv')
    # submission = pd.read_csv('data/sample_submission.csv')
    print("데이터 로드 완료")
except FileNotFoundError:
    print("data/ 폴더에 train.csv, test.csv, sample_submission.csv 파일이 있는지 확인해주세요.")

## 2. 데이터 전처리
- 결측치 처리 (occyp_type)
- 바이너리 변수 변환 (0, 1)
- 명목형 변수 원핫 인코딩
- 날짜 데이터 변환 (음수 -> 양수, 연단위 변환)

In [ ]:
def preprocess_data(df):
    # 결측치 처리: occyp_type (직업 유형)의 결측치를 'None'으로 대체
    df['occyp_type'] = df['occyp_type'].fillna('None')
    
    # DAYS_BIRTH -> age (나이)
    df['age'] = abs(df['DAYS_BIRTH']) // 365
    
    # DAYS_EMPLOYED -> worked_years (근무 연수)
    # 양수(365243)는 무직을 의미하므로 0으로 처리
    df['worked_years'] = df['DAYS_EMPLOYED'].apply(lambda x: 0 if x > 0 else abs(x) // 365)
    
    # begin_month -> 발행된지 몇 달 되었는지 (발행 개월)
    df['begin_month'] = abs(df['begin_month'])
    
    # 불필요한 컬럼 삭제
    df = df.drop(['index', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL'], axis=1, errors='ignore')
    
    return df

In [ ]:
if 'train' in locals():
    train_df = preprocess_data(train)
    test_df = preprocess_data(test)
    
    # Label Encoding / One-Hot Encoding
    from sklearn.preprocessing import LabelEncoder
    
    cat_cols = train_df.select_dtypes(include='object').columns
    
    for col in cat_cols:
        le = LabelEncoder()
        # train과 test를 합쳐서 fitting
        combined = pd.concat([train_df[col], test_df[col]], axis=0)
        le.fit(combined)
        train_df[col] = le.transform(train_df[col])
        test_df[col] = le.transform(test_df[col])
    
    print("전처리 및 인코딩 완료")

## 3. 모델링 (RandomForest Baseline)
중복 데이터를 방지하기 위해 Stratified K-Fold를 사용합니다.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

if 'train_df' in locals():
    X = train_df.drop('credit', axis=1)
    y = train_df['credit']
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    rf_models = []
    log_losses = []
    
    for i, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"--- Fold {i+1} --- ")
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        
        # 예측 (확률값)
        y_pred_proba = model.predict_proba(X_val)
        loss = log_loss(y_val, y_pred_proba)
        
        rf_models.append(model)
        log_losses.append(loss)
        print(f"Log Loss: {loss:.4f}")
        
    print(f"평균 Log Loss: {np.mean(log_losses):.4f}")

## 4. 최종 예측 및 제출 파일 생성

In [ ]:
if rf_models:
    final_preds = np.zeros((test_df.shape[0], 3)) # credit 0, 1, 2 에 대한 분류
    
    for model in rf_models:
        final_preds += model.predict_proba(test_df) / len(rf_models)
    
    submission.iloc[:, 1:] = final_preds
    submission.to_csv('output/baseline_submission.csv', index=False)
    print("제출 파일 저장 완료: output/baseline_submission.csv")